# Predicting Judicial Decisions of the European Court of Human Rights

**Author:** Zoé de Vries  
**Affiliation:** ENS Paris-Saclay — ARIA research programme  
**Date:** 2023  

## Overview

This notebook implements a text classification pipeline to predict whether the European Court of Human Rights (ECHR) has found a violation of a given article of the European Convention on Human Rights.

The approach follows Medvedeva et al. (2019, 2022): case documents are parsed by structural section, represented as TF-IDF vectors, and classified using a Linear Support Vector Machine (LinearSVC). Hyperparameters were determined per article using exhaustive grid search (see `grid_search_exp1.py`) and are fixed here for reproducibility.

**Dataset:** ECtHR Crystal Ball — Medvedeva, Vols & Wieling (2019)  
Available at: https://github.com/masha-medvedeva/ECtHR_crystal_ball

**Evaluation:** 10-fold cross-validation per article. Metrics: accuracy, precision, recall, F1 (macro), confusion matrix.

---

## Setup

Before running, set `DATA_PATH` in the next cell to the local path of your `crystal_ball_data/` folder.  
See the README for dataset download instructions.

In [ ]:
# ============================================================
# CONFIGURATION — edit this cell before running
# ============================================================

# Path to the crystal_ball_data/ folder from the ECtHR Crystal Ball dataset.
# Clone the dataset with:
#   git clone https://github.com/masha-medvedeva/ECtHR_crystal_ball.git
# then set DATA_PATH to the crystal_ball_data/ subdirectory.
DATA_PATH = '../../crystal_ball_data/'  # <-- update this path

# Articles to process. All nine articles from the dataset are included by default.
ARTICLES = ['Article2', 'Article3', 'Article5', 'Article6',
             'Article8', 'Article10', 'Article11', 'Article13', 'Article14']

## 1. Imports

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
import glob, re, os, sys, random
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, precision_recall_fscore_support)
from nltk.corpus import stopwords
from random import shuffle

## 2. Text extraction

ECHR case documents follow a standard structure with clearly labelled sections (PROCEDURE, THE FACTS, CIRCUMSTANCES OF THE CASE, RELEVANT LAW/DOMESTIC LAW, THE LAW). The functions below extract text between two section-boundary markers using regex.

Each document is also parsed for its dateline (used to record the case year, though year filtering is not applied in this pipeline).

**Sections available:**
- `facts` — THE FACTS → THE LAW
- `circumstances` — CIRCUMSTANCES → RELEVANT
- `relevant_law` — RELEVANT → THE LAW or PROCEEDINGS
- `procedure` — PROCEDURE → THE FACTS
- `procedure+facts` — PROCEDURE → THE LAW (combines procedure and facts sections)

In [ ]:
def extract_text(starts, ends, cases, violation):
    """
    Extract a text section from a list of case files.

    Reads each file, identifies the section starting with `starts` and
    ending at `ends`, and returns a list of (text, label, year) tuples.

    Parameters
    ----------
    starts : str
        Regex pattern marking the start of the target section.
    ends : str
        Regex pattern marking the end of the target section.
    cases : list of str
        File paths to case documents.
    violation : str
        Label for this set of cases: 'violation' or 'non-violation'.

    Returns
    -------
    list of tuple
        Each tuple is (text: str, label: str, year: int).
        Cases without a parseable dateline are silently skipped.
    """
    facts = []
    D = []
    years = []
    for case in cases:
        contline = ''
        year = 0
        with open(case, 'r') as f:
            for line in f:
                dat = re.search(r'^([0-9]{1,2}\s\w+\s([0-9]{4}))', line)
                if dat is not None:
                    year = int(dat.group(2))
                    break
            if year > 0:
                years.append(year)
                wr = 0
                for line in f:
                    if wr == 0:
                        if re.search(starts, line) is not None:
                            wr = 1
                    if wr == 1 and re.search(ends, line) is None:
                        contline += line
                        contline += '\n'
                    elif re.search(ends, line) is not None:
                        break
                facts.append(contline)
    for i in range(len(facts)):
        D.append((facts[i], violation, years[i]))
    return D

In [ ]:
def extract_parts(train_path, violation, part):
    """
    Extract a named structural section from all case files matching a glob pattern.

    Parameters
    ----------
    train_path : str
        Glob pattern for case files, e.g. 'data/train/Article6/violation/*.txt'.
    violation : str
        Label: 'violation' or 'non-violation'.
    part : str
        Section name. One of: 'facts', 'circumstances', 'relevant_law',
        'procedure', 'procedure+facts'.

    Returns
    -------
    list of tuple
        Each tuple is (text: str, label: str, year: int).
    """
    cases = glob.glob(train_path)
    facts = []
    D = []
    years = []

    if part == 'relevant_law':
        for case in cases:
            year = 0
            contline = ''
            with open(case, 'r') as f:
                for line in f:
                    dat = re.search(r'^([0-9]{1,2}\s\w+\s([0-9]{4}))', line)
                    if dat is not None:
                        year = int(dat.group(2))
                        break
                if year > 0:
                    years.append(year)
                    wr = 0
                    for line in f:
                        if wr == 0:
                            if re.search('RELEVANT', line) is not None:
                                wr = 1
                        if wr == 1 and re.search('THE LAW', line) is None and re.search('PROCEEDINGS', line) is None:
                            contline += line
                            contline += '\n'
                        elif re.search('THE LAW', line) is not None or re.search('PROCEEDINGS', line) is not None:
                            break
                    facts.append(contline)
        for i in range(len(facts)):
            D.append((facts[i], violation, years[i]))

    elif part == 'facts':
        D = extract_text('THE FACTS', 'THE LAW', cases, violation)
    elif part == 'circumstances':
        D = extract_text('CIRCUMSTANCES', 'RELEVANT', cases, violation)
    elif part == 'procedure':
        D = extract_text('PROCEDURE', 'THE FACTS', cases, violation)
    elif part == 'procedure+facts':
        D = extract_text('PROCEDURE', 'THE LAW', cases, violation)
    else:
        raise ValueError(f"Unknown part: '{part}'. Choose from: facts, circumstances, relevant_law, procedure, procedure+facts")

    return D

## 3. Model training and evaluation

The model is a `LinearSVC` wrapped in a `sklearn.pipeline.Pipeline` together with TF-IDF vectorisation. Evaluation uses 10-fold cross-validation (`cross_val_predict`), which ensures all cases are used for both training and testing across folds.

In [ ]:
def train_model_cross_val(Xtrain, Ytrain, vec, c):
    """
    Train and evaluate a LinearSVC pipeline using 10-fold cross-validation.

    Parameters
    ----------
    Xtrain : list of str
        Input texts.
    Ytrain : list of str
        Labels ('violation' or 'non-violation').
    vec : tuple
        A (name, TfidfVectorizer) tuple as expected by FeatureUnion.
    c : float
        Regularisation parameter for LinearSVC.
    """
    print('*** 10-fold cross-validation ***')
    pipeline = Pipeline([
        ('features', FeatureUnion([vec])),
        ('classifier', LinearSVC(C=c))
    ])
    Ypredict = cross_val_predict(pipeline, Xtrain, Ytrain, cv=10)
    evaluate(Ytrain, Ypredict)


def train_model_test(Xtrain, Ytrain, Xtest_v, Ytest_v, vec, c):
    """
    Train a LinearSVC pipeline on the training set and evaluate on a held-out test set.

    Parameters
    ----------
    Xtrain, Ytrain : list
        Training data and labels.
    Xtest_v, Ytest_v : list
        Test data and labels (violation cases only).
    vec : tuple
        A (name, TfidfVectorizer) tuple.
    c : float
        Regularisation parameter for LinearSVC.
    """
    pipeline = Pipeline([
        ('features', FeatureUnion([vec])),
        ('classifier', LinearSVC(C=c))
    ])
    pipeline.fit(Xtrain, Ytrain)
    print('*** Testing on violation test set ***')
    Ypredict = pipeline.predict(Xtest_v)
    evaluate(Ytest_v, Ypredict)


def evaluate(Ytest, Ypredict):
    """
    Print accuracy, classification report, macro-averaged P/R/F1, and confusion matrix.

    Parameters
    ----------
    Ytest : list of str
        True labels.
    Ypredict : list of str
        Predicted labels.
    """
    print('Accuracy:', accuracy_score(Ytest, Ypredict))
    print('\nClassification report:\n', classification_report(Ytest, Ypredict))
    print('\nMacro P/R/F1:', precision_recall_fscore_support(Ytest, Ypredict, average='macro'))
    print('\nConfusion matrix:\n', confusion_matrix(Ytest, Ypredict))
    print('\n' + '_' * 40 + '\n')

## 4. Pipeline runner

Loads training data from the appropriate article subfolder, shuffles cases, and runs cross-validation.

In [ ]:
def run_pipeline(article, part, vec, c, path):
    """
    Load data for one article and run cross-validated classification.

    Parameters
    ----------
    article : str
        Article name, e.g. 'Article6'.
    part : str
        Text section to use for training.
    vec : tuple
        (name, TfidfVectorizer) tuple.
    c : float
        LinearSVC regularisation parameter.
    path : str
        Root path to the crystal_ball_data/ directory.
    """
    print(f'Trained on *{part}* section | Article: {article}')

    v  = extract_parts(path + 'train/' + article + '/violation/*.txt',     'violation',     part)
    nv = extract_parts(path + 'train/' + article + '/non-violation/*.txt', 'non-violation', part)
    trainset = v + nv
    shuffle(trainset)

    Xtrain = [i[0] for i in trainset]
    Ytrain = [i[1] for i in trainset]

    # Test set (violation cases only, used for held-out evaluation if needed)
    if article == 'Article14':
        test = extract_parts(path + 'test_violations/' + article + '/*.txt', 'non-violation', part)
    else:
        test = extract_parts(path + 'test_violations/' + article + '/*.txt', 'violation', part)
    Xtest_v = [i[0] for i in test]
    Ytest_v = [i[1] for i in test]

    print(f'Training on {Ytrain.count("violation")} violation + '
          f'{Ytrain.count("non-violation")} non-violation = '
          f'{len(Ytrain)} cases')
    print(f'Test cases available (violation): {len(Xtest_v)}')

    train_model_cross_val(Xtrain, Ytrain, vec, c)

## 5. Run all articles

Parameters per article were determined by exhaustive grid search (`grid_search_exp1.py`).  
They are fixed here for reproducibility.

| Article    | Section          | ngram range | binary | IDF   | norm | min_df | stop words | C   |
|------------|-----------------|-------------|--------|-------|------|--------|------------|-----|
| Article 2  | procedure+facts | (3,4)       | False  | True  | l2   | 2      | None       | 0.1 |
| Article 3  | facts           | (1,1)       | True   | True  | None | 1      | None       | 0.1 |
| Article 5  | facts           | (1,1)       | True   | True  | l2   | 3      | None       | 1   |
| Article 6  | procedure+facts | (2,4)       | True   | True  | l2   | 2      | None       | 5   |
| Article 8  | facts           | (3,3)       | True   | False | l2   | 1      | None       | 1   |
| Article 10 | procedure+facts | (1,1)       | False  | False | l2   | 1      | None       | 5   |
| Article 11 | procedure       | (1,1)       | False  | False | l1   | 2      | english    | 1   |
| Article 13 | procedure+facts | (1,2)       | False  | True  | l2   | 1      | None       | 5   |
| Article 14 | procedure+facts | (1,1)       | True   | True  | l2   | 3      | english    | 5   |

In [ ]:
# Article-specific parameters (from grid search)
PARAMS = {
    'Article2':  {'part': 'procedure+facts', 'c': 0.1,
                  'vec': TfidfVectorizer(analyzer='word', ngram_range=(3,4),  binary=False, lowercase=True,  min_df=2, norm='l2',  stop_words=None,      use_idf=True)},
    'Article3':  {'part': 'facts',           'c': 0.1,
                  'vec': TfidfVectorizer(analyzer='word', ngram_range=(1,1),  binary=True,  lowercase=True,  min_df=1, norm=None,  stop_words=None,      use_idf=True)},
    'Article5':  {'part': 'facts',           'c': 1,
                  'vec': TfidfVectorizer(analyzer='word', ngram_range=(1,1),  binary=True,  lowercase=True,  min_df=3, norm='l2',  stop_words=None,      use_idf=True)},
    'Article6':  {'part': 'procedure+facts', 'c': 5,
                  'vec': TfidfVectorizer(analyzer='word', ngram_range=(2,4),  binary=True,  lowercase=True,  min_df=2, norm='l2',  stop_words=None,      use_idf=True)},
    'Article8':  {'part': 'facts',           'c': 1,
                  'vec': TfidfVectorizer(analyzer='word', ngram_range=(3,3),  binary=True,  lowercase=True,  min_df=1, norm='l2',  stop_words=None,      use_idf=False)},
    'Article10': {'part': 'procedure+facts', 'c': 5,
                  'vec': TfidfVectorizer(analyzer='word', ngram_range=(1,1),  binary=False, lowercase=False, min_df=1, norm='l2',  stop_words=None,      use_idf=False)},
    'Article11': {'part': 'procedure',       'c': 1,
                  'vec': TfidfVectorizer(analyzer='word', ngram_range=(1,1),  binary=False, lowercase=True,  min_df=2, norm='l1',  stop_words='english', use_idf=False)},
    'Article13': {'part': 'procedure+facts', 'c': 5,
                  'vec': TfidfVectorizer(analyzer='word', ngram_range=(1,2),  binary=False, lowercase=True,  min_df=1, norm='l2',  stop_words=None,      use_idf=True)},
    'Article14': {'part': 'procedure+facts', 'c': 5,
                  'vec': TfidfVectorizer(analyzer='word', ngram_range=(1,1),  binary=True,  lowercase=True,  min_df=3, norm='l2',  stop_words='english', use_idf=True)},
}

for article in ARTICLES:
    print('=' * 50)
    print(article)
    print('=' * 50)
    p = PARAMS[article]
    run_pipeline(
        article=article,
        part=p['part'],
        vec=('wordvec', p['vec']),
        c=p['c'],
        path=DATA_PATH
    )